# DROP evaluation on Colab — Qwen 2.5 14B, 4 conditions

Runs **DROP-only** evaluation with:
- **Model:** Qwen/Qwen2.5-14B-Instruct
- **Conditions:** direct, short, medium, long
- **max_samples:** 100 (validation split)
- **temperature:** 0

**Note:** 14B in bfloat16 needs ~28GB GPU memory. Use **Runtime → Change runtime type → GPU**. T4 may OOM; A100 or High-RAM helps. For free-tier T4, use `Qwen/Qwen2.5-7B-Instruct` in the config cell below.

In [ ]:
# Install dependencies (run once)
!pip install -q torch transformers datasets accelerate

In [ ]:
# Config
MODEL_NAME = "Qwen/Qwen2.5-14B-Instruct"  # or "Qwen/Qwen2.5-7B-Instruct" for smaller GPU
SPLIT = "validation"
MAX_SAMPLES = 100
SEED = 42
TEMPERATURE = 0.0
DTYPE = "bfloat16"  # or "float16" if bfloat16 unsupported
RESULTS_DIR = "results_large_drop"  # outputs under this folder

CONDITIONS = ["direct", "short", "medium", "long"]
COND_CFG = {
    "direct": {"steps": 0, "max_new_tokens": 64},
    "short":  {"steps": 2, "max_new_tokens": 128},
    "medium": {"steps": 4, "max_new_tokens": 256},
    "long":   {"steps": 8, "max_new_tokens": 512},
}

import os
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"GPU: {os.environ.get('COLAB_GPU', 'N/A')}; Results -> {RESULTS_DIR}/")

In [ ]:
# DROP scoring (normalize + Exact Match)
import re
import string
from typing import List, Optional

_ARTICLES_RE = re.compile(r"\b(a|an|the)\b", flags=re.IGNORECASE)
_WHITESPACE_RE = re.compile(r"\s+")
_PUNCT_TABLE = str.maketrans({p: " " for p in string.punctuation})

def _strip_quotes(s: str) -> str:
    s = s.strip()
    if (s.startswith('"') and s.endswith('"')) or (s.startswith("'") and s.endswith("'")):
        s = s[1:-1].strip()
    return s

def normalize_answer(text: str) -> str:
    if text is None:
        return ""
    text = _strip_quotes(text)
    text = text.lower()
    if re.fullmatch(r"[-+]?\d+(\.\d+)?", text.strip()):
        x = float(text)
        return str(int(x)) if x.is_integer() else str(x).rstrip("0").rstrip(".")
    text = text.translate(_PUNCT_TABLE)
    text = _ARTICLES_RE.sub(" ", text)
    text = _WHITESPACE_RE.sub(" ", text).strip()
    return text

def em_over_ground_truths(pred: str, golds: List[str]) -> bool:
    if pred is None:
        pred = ""
    if not golds:
        return False
    p = normalize_answer(pred)
    for g in golds:
        if p == normalize_answer(g):
            return True
    return False

def clean_final_answer_span(ans: str) -> str:
    if ans is None:
        return ""
    ans = ans.strip()
    ans = ans.splitlines()[0].strip()
    return ans

def extract_final_answer(text: str) -> Optional[str]:
    if not text:
        return None
    m = re.search(r"Final Answer\s*:\s*(.+)", text, flags=re.IGNORECASE)
    if m:
        return clean_final_answer_span(m.group(1))
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    if not lines:
        return None
    return clean_final_answer_span(lines[-1])

print("Scoring functions defined.")

In [ ]:
# Dataset loading and prompt helpers
import random
import json
import time
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

from datasets import load_dataset

SYSTEM_PROMPT = "You are a careful reading comprehension solver. Follow formatting rules exactly."
USER_PROMPT_DIRECT = """Answer the question based ONLY on the given passage.

Rules:
- Do NOT show any reasoning.
- Output exactly ONE line in the format:
  Final Answer: <answer>
- <answer> should be a short span or a number, copied from or computed from the passage.
- Do not output anything else.

Passage:
{passage}

Question:
{question}
"""
USER_PROMPT_COT = """Answer the question based ONLY on the given passage.

Rules:
- Write EXACTLY {n_steps} reasoning steps.
- Each step MUST be a single line and MUST start with "Step k:" (k = 1..{n_steps}).
- Each step MUST be one short sentence (max 18 words).
- After the last step, output exactly ONE final line:
  Final Answer: <answer>
- <answer> should be a short span or a number, copied from or computed from the passage.
- Do not output anything else.

Passage:
{passage}

Question:
{question}
"""

def _extract_gold_answers(ex: Dict[str, Any]) -> List[str]:
    def _nonempty_str_list(xs):
        return [str(x).strip() for x in xs if str(x).strip()]
    def _from_answer_dict(a: Dict[str, Any]) -> List[str]:
        out = []
        for k in ("spans", "text", "answers"):
            if k in a and isinstance(a[k], list):
                out.extend(_nonempty_str_list(a[k]))
        num = a.get("number")
        if isinstance(num, str) and num.strip():
            out.append(num.strip())
        date = a.get("date")
        if isinstance(date, dict):
            parts = [str(date.get(k)).strip() for k in ("month", "day", "year") if date.get(k)]
            if parts:
                out.append(" ".join(parts))
        elif isinstance(date, str) and date.strip():
            out.append(date.strip())
        seen = set()
        return [x for x in out if x not in seen and not seen.add(x)]
    if "answers" in ex and isinstance(ex["answers"], dict):
        out = _from_answer_dict(ex["answers"])
        if out:
            return out
    # HuggingFace ucinlp/drop: answers_spans = {"spans": ["ans1", ...]}
    if "answers_spans" in ex:
        v = ex["answers_spans"]
        if isinstance(v, dict) and "spans" in v:
            spans = v["spans"]
            try:
                out = _nonempty_str_list(list(spans)) if spans is not None else []
            except (TypeError, ValueError):
                out = [str(spans).strip()] if str(spans).strip() else []
            if out:
                return out
    for key in ("gold_answers", "answer_text"):
        if key in ex:
            v = ex[key]
            if isinstance(v, list):
                out = _nonempty_str_list(v)
                if out:
                    return out
            if isinstance(v, str) and v.strip():
                return [v.strip()]
    if "answer" in ex:
        v = ex["answer"]
        if isinstance(v, list):
            out = _nonempty_str_list(v)
            if out:
                return out
        if isinstance(v, str) and v.strip():
            return [v.strip()]
    return []

def load_drop(split: str) -> List[Dict[str, Any]]:
    ds = load_dataset("ucinlp/drop", split=split)
    return [dict(x) for x in ds]

def pick_indices(n: int, max_samples: Optional[int], seed: int, shuffle: bool) -> List[int]:
    idx = list(range(n))
    if shuffle:
        rnd = random.Random(seed)
        rnd.shuffle(idx)
    if max_samples is not None and max_samples > 0:
        idx = idx[:max_samples]
    return idx

def build_user_prompt(passage: str, question: str, condition: str, n_steps: int) -> str:
    if condition == "direct":
        return USER_PROMPT_DIRECT.format(passage=passage, question=question)
    return USER_PROMPT_COT.format(passage=passage, question=question, n_steps=n_steps)

def format_chat_if_possible(tokenizer, system_prompt: str, user_prompt: str, use_chat_template: bool = True) -> str:
    if use_chat_template and hasattr(tokenizer, "apply_chat_template"):
        messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
        try:
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            pass
    return f"{system_prompt}\n\n{user_prompt}".strip()

def estimate_steps(output_text: str) -> int:
    return len(re.findall(r"\bStep\s+\d+\s*:", output_text)) if output_text else 0

rows = load_drop(SPLIT)
idxs = pick_indices(len(rows), MAX_SAMPLES, SEED, shuffle=True)
print(f"Loaded DROP {SPLIT}: {len(rows)} total, running on {len(idxs)} samples.")

In [ ]:
# Load model once (shared across all 4 conditions)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

@dataclass
class RuntimeCfg:
    temperature: float
    top_p: float
    max_new_tokens: int
    use_chat_template: bool

def load_model(model_name: str, dtype: str):
    tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    if tok.pad_token_id is None and tok.eos_token_id is not None:
        tok.pad_token_id = tok.eos_token_id
    torch_dtype = {"float16": torch.float16, "bfloat16": torch.bfloat16, "float32": torch.float32}.get(dtype.lower())
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch_dtype,
        device_map="auto",
    )
    model.eval()
    return model, tok

def generate(model, tokenizer, prompt: str, cfg: RuntimeCfg) -> Tuple[str, int, int]:
    enc = tokenizer(prompt, return_tensors="pt", truncation=True, padding=False)
    input_ids = enc["input_ids"].to(model.device)
    attn = enc.get("attention_mask")
    if attn is not None:
        attn = attn.to(model.device)
    input_tokens = int(input_ids.shape[-1])
    with torch.no_grad():
        out = model.generate(
            input_ids=input_ids,
            attention_mask=attn,
            do_sample=(cfg.temperature > 0),
            temperature=cfg.temperature if cfg.temperature > 0 else None,
            top_p=cfg.top_p,
            max_new_tokens=cfg.max_new_tokens,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    gen_ids = out[0, input_tokens:]
    output_tokens = int(gen_ids.shape[-1])
    text = tokenizer.decode(gen_ids, skip_special_tokens=True)
    return text, input_tokens, output_tokens

def model_tag(model_name: str) -> str:
    return model_name.split("/")[-1].lower().replace(" ", "_").replace(".", "_")

print("Loading model (this may take a few minutes)...")
model, tokenizer = load_model(MODEL_NAME, DTYPE)
print("Model loaded.")

In [ ]:
# Run evaluation for all 4 conditions and save to RESULTS_DIR
tag = model_tag(MODEL_NAME)
for condition in CONDITIONS:
    n_steps = COND_CFG[condition]["steps"]
    max_new_tokens = COND_CFG[condition]["max_new_tokens"]
    out_name = f"{tag}_{condition}_{SPLIT}_n{MAX_SAMPLES}.jsonl"
    out_path = os.path.join(RESULTS_DIR, out_name)
    run_cfg = RuntimeCfg(temperature=TEMPERATURE, top_p=1.0, max_new_tokens=max_new_tokens, use_chat_template=True)
    correct_cnt = 0
    t0 = time.time()
    with open(out_path, "w", encoding="utf-8") as f:
        for k, i in enumerate(idxs, start=1):
            ex = rows[i]
            passage = ex.get("passage") or ex.get("context") or ""
            question = ex.get("question") or ""
            gold_answers = _extract_gold_answers(ex)
            user_prompt = build_user_prompt(passage, question, condition, n_steps=n_steps)
            prompt = format_chat_if_possible(tokenizer, SYSTEM_PROMPT, user_prompt, True)
            start = time.time()
            gen_text, in_tok, out_tok = generate(model, tokenizer, prompt, run_cfg)
            steps_est = estimate_steps(gen_text)
            if condition != "direct" and steps_est != n_steps:
                gen_text, in_tok, out_tok = generate(model, tokenizer, prompt, run_cfg)
            latency = time.time() - start
            pred_ans = extract_final_answer(gen_text) or ""
            ok = em_over_ground_truths(pred_ans, gold_answers)
            correct_cnt += int(ok)
            rec = {
                "dataset": "drop", "split": SPLIT, "qid": i, "model_name": MODEL_NAME, "condition": condition,
                "steps_target": n_steps, "max_new_tokens": max_new_tokens,
                "passage": passage, "question": question, "gold_answers": gold_answers,
                "pred_raw": gen_text, "pred_answer": pred_ans, "em": bool(ok), "correct": bool(ok),
                "input_tokens": in_tok, "output_tokens": out_tok, "steps_est": estimate_steps(gen_text),
                "latency_sec": latency, "seed": SEED,
            }
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            if k % 25 == 0 or k == len(idxs):
                print(f"[{condition}] {k}/{len(idxs)} em={correct_cnt/k:.3f}")
    total = time.time() - t0
    print(f"[done] {condition} -> {out_path} em={correct_cnt/len(idxs):.4f} time={total:.1f}s")
print("All 4 conditions finished. Outputs in", RESULTS_DIR)